# DataLab — Quickstart & Stack Verification

This notebook verifies that the entire local lakehouse stack is working:
- PySpark connecting to the Spark Standalone cluster
- Delta Lake read/write
- MinIO (S3-compatible) storage via `s3a://`
- SQL queries on Delta tables

Run all cells top-to-bottom. If the last cell succeeds, the stack is healthy.

## 1. Create SparkSession

In [3]:
import os
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

SPARK_MASTER = os.environ.get("SPARK_MASTER", "spark://spark-master:7077")
MINIO_ENDPOINT = os.environ.get("MINIO_ENDPOINT", "http://minio:9000")
MINIO_ACCESS_KEY = os.environ.get("MINIO_ACCESS_KEY", "minioadmin")
MINIO_SECRET_KEY = os.environ.get("MINIO_SECRET_KEY", "minioadmin")

builder = (
    SparkSession.builder
    .appName("DataLab-Quickstart")
    .master(SPARK_MASTER)
    # MinIO / S3A config
    .config("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("WARN")

print(f"Spark version : {spark.version}")
print(f"Master        : {spark.sparkContext.master}")
print(f"App UI        : {spark.sparkContext.uiWebUrl}")

Spark version : 3.5.1
Master        : spark://spark-master:7077
App UI        : http://3d281ca878ae:4040


## 2. Write a Delta table to MinIO (Bronze layer)

In [4]:
from pyspark.sql import Row
from datetime import date

BRONZE_PATH = "s3a://warehouse/bronze/test_table"

sample_data = [
    Row(id=1, name="Alice",   value=100.0, event_date=date(2024, 1, 1)),
    Row(id=2, name="Bob",     value=200.5, event_date=date(2024, 1, 2)),
    Row(id=3, name="Charlie", value=150.0, event_date=date(2024, 1, 3)),
    Row(id=4, name="Diana",   value=300.0, event_date=date(2024, 1, 4)),
]

df = spark.createDataFrame(sample_data)

(
    df.write
    .format("delta")
    .mode("overwrite")
    .save(BRONZE_PATH)
)

print(f"Written {df.count()} rows to {BRONZE_PATH}")

26/03/01 11:55:18 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Written 4 rows to s3a://warehouse/bronze/test_table


## 3. Read the Delta table back

In [5]:
df_read = spark.read.format("delta").load(BRONZE_PATH)

print("Schema:")
df_read.printSchema()

print("Data:")
df_read.show()

Schema:
root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- value: double (nullable = true)
 |-- event_date: date (nullable = true)

Data:


+---+-------+-----+----------+
| id|   name|value|event_date|
+---+-------+-----+----------+
|  3|Charlie|150.0|2024-01-03|
|  4|  Diana|300.0|2024-01-04|
|  1|  Alice|100.0|2024-01-01|
|  2|    Bob|200.5|2024-01-02|
+---+-------+-----+----------+



## 4. SQL query on the Delta table

In [6]:
# Register as a temp view and run SQL
df_read.createOrReplaceTempView("bronze_test")

result = spark.sql("""
    SELECT
        name,
        value,
        event_date,
        RANK() OVER (ORDER BY value DESC) AS value_rank
    FROM bronze_test
    ORDER BY value_rank
""")

result.show()

+-------+-----+----------+----------+
|   name|value|event_date|value_rank|
+-------+-----+----------+----------+
|  Diana|300.0|2024-01-04|         1|
|    Bob|200.5|2024-01-02|         2|
|Charlie|150.0|2024-01-03|         3|
|  Alice|100.0|2024-01-01|         4|
+-------+-----+----------+----------+



26/03/01 11:56:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/01 11:56:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/01 11:56:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/01 11:56:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/01 11:56:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


## 5. Delta Lake table history (like DESCRIBE HISTORY in Databricks)

In [7]:
from delta.tables import DeltaTable

dt = DeltaTable.forPath(spark, BRONZE_PATH)
dt.history().select("version", "timestamp", "operation", "operationParameters").show(truncate=False)

+-------+-------------------+---------+--------------------------------------+
|version|timestamp          |operation|operationParameters                   |
+-------+-------------------+---------+--------------------------------------+
|1      |2026-03-01 11:55:22|WRITE    |{mode -> Overwrite, partitionBy -> []}|
|0      |2026-03-01 11:54:57|WRITE    |{mode -> Overwrite, partitionBy -> []}|
+-------+-------------------+---------+--------------------------------------+



## 6. Stack summary

In [ ]:
import importlib.metadata
import pyspark
import pandas as pd
import pyarrow as pa

print("=" * 50)
print("DataLab stack is HEALTHY")
print("=" * 50)
print(f"  PySpark      : {pyspark.__version__}")
print(f"  Delta Lake   : {importlib.metadata.version('delta-spark')}")
print(f"  Pandas       : {pd.__version__}")
print(f"  PyArrow      : {pa.__version__}")
print(f"  Spark master : {spark.sparkContext.master}")
print("")
print("UIs:")
print("  Spark Master   → http://localhost:8080")
print("  Spark Worker   → http://localhost:8081")
print("  History Server → http://localhost:18080")
print("  MinIO Console  → http://localhost:9001")
print("  JupyterLab     → http://localhost:8888")